In [1]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/reasoning/sft/2025-05-16 16:27:44/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())

state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)

/root/micromamba/envs/fla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/root/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


<All keys matched successfully>

In [2]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(dtype=dtype, device=device)

In [3]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=512,
    do_sample=True,
    top_k=20,
    top_p=0.7,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.1,
    use_cache=True
)

In [6]:
prompt = [
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot>"
]
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=True,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>Transformer（Generative Transformer）是一种用于生成、训练和生成自然语言处理的机器学习模型。它通过在大量数据上进行训练，并在生成过程中生成符合条件的输出。
Transformer架构是指一种用于生成、训练和生成自然语言处理模型的工具。它通过构建一个神经网络模型，使得模型能够生成人类可读的文本、图像、语音和文本内容。
下面是一些常见的Transformer架构的例子：
1. 预训练模型：通常使用预训练模型来生成自然语言文本。这可能需要大量的数据和计算资源，但是通常需要大量的数据和计算资源。
2. 模型训练：使用预训练模型来生成自然语言文本。这可能需要大量的数据和计算资源，但对于大规模的数据和计算资源来说可能需要大量的计算资源。
3. 模型评估：使用训练数据来评估模型的性能。这可能需要大量的数据和计算资源，但是对于大规模的数据和计算资源来说可能需要大量的计算资源。
4. 模型评估：使用训练好的模型来评估模型的性能。这可能需要大量的计算资源，但对于大规模的数据和计算资源来说可能需要大量的计算资源。
5. 模型部署：使用训练好的模型来训练模型。这可能需要大量的计算资源，但对于大规模的数据和计算资源来说可能需要大量的计算资源。
6. 模型部署：使用训练好的模型来训练模型。这可能需要大量的计算资源，但对于大规模的数据和计算资源来说可能需要大量的计算资源。
总之，Transformer架构是一种基于Transformer架构的机器学习模型，它能够自动地生成自然语言文本、语音和文本内容。</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pa

In [7]:
prompt = [
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot><think>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot><think>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot><think>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot><think>"
]
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=True,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot><think>：在神经网络中，Transformer架构是一种基于神经网络的架构。它通过将输入和输出数据映射到输出层，从而实现类似于人类感知和学习的能力。

Transformer架构是一种基于神经网络的架构，它使用多个层（卷积层、池化层、全连接层）来处理数据。每个层都有多个权重，这些权重由权重和池化层组成。

具体来说，Transformer架构可以表示为：

1. **输入层**：输入层负责处理输入数据，每个神经元都会生成一个输出层。
2. **输出层**：输入层负责处理输入数据，每个神经元都会生成一个输出层。
3. **全连接层**：全连接层负责处理输入数据，每个神经元都会生成一个输出层。

具体来说，Transformer架构通常使用多个层来处理输入数据。每个层都包含一个输出层，每个神经元都有一个输出层。

下面是一些具体的步骤：

1. **定义神经网络结构**：
   - 定义一个神经网络，用于处理输入数据。
   - 定义一个神经网络，用于处理输入数据。
   - 定义一个激活函数，用于激活输入数据。

2. **定义Transformer架构**：
   - 定义一个卷积神经网络，用于处理输入数据。
   - 定义一个激活函数，用于激活输入数据。
   - 定义一个激活函数，用于激活输入数据。

3. **定义Transformer架构**：
   - 定义一个Transformer架构，用于处理输入数据。
   - 定义一个激活函数，用于激活输入数据。
   - 定义一个激活函数，用于激活输入数据。

4. **训练模型**：
   - 训练模型：使用预训练的Transformer模型，如GPT-2、BERT等。
   - 模型训练：使用训练数据

In [5]:
generate_config.max_new_tokens = 4096
prompt = [
    "<s><user>A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie. You meet 3 inhabitants: Olivia, Amelia, and James. \"James is a knave\" - Olivia. Amelia expressed that Olivia is a knave. James stated, \"James is a knight if and only if Amelia is a knight\". So who is a knight and who is a knave?</s>\n<s><bot><think>"
]
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=True,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie. You meet 3 inhabitants: Olivia, Amelia, and James. "James is a knave" - Olivia. Amelia expressed that Olivia is a knave. James stated, "James is a knight if and only if Amelia is a knight". So who is a knight and who is a knave?</s>
<s><bot><think>Okay, let's try to figure out this knights and knaves puzzle. So we have three people: Olivia, Amelia, and James. Each of them made a statement. Let's break down each statement and see what we can deduce.

First, Olivia says that Olivia is a knave. So if Olivia is a knight, then her statement is true, which means Olivia is a knave. But if Olivia is a knave, then her statement is false, so Olivia would be a knight. But if Olivia is a knave, then her statement is false, so Olivia is a knave. That's a contradiction. So Olivia can't be a knight. Therefore, Olivia must be a knave. Because if she's a knave, her statement 